## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 14: Modell-Deployment
# Niveau: Fortgeschrittene
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json
import os
import time
from datetime import datetime

np.random.seed(42)
tf.random.set_seed(42)

print("=== Modell-Versionierung ===\n")

# ── 1. Daten laden ─────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

# ── 2. Modell-Registry ────────────────────────────────────
MODEL_REGISTRY_DIR = 'model_registry/'
METADATA_FILE      = os.path.join(MODEL_REGISTRY_DIR, 'registry.json')

os.makedirs(MODEL_REGISTRY_DIR, exist_ok=True)

class ModelRegistry:
    """Verwaltet Modell-Versionen mit Metadaten."""

    def __init__(self, registry_dir, metadata_file):
        self.registry_dir  = registry_dir
        self.metadata_file = metadata_file
        self.registry      = self._load_registry()

    def _load_registry(self):
        if os.path.exists(self.metadata_file):
            with open(self.metadata_file, 'r') as f:
                return json.load(f)
        return {'models': [], 'best_model': None}

    def _save_registry(self):
        with open(self.metadata_file, 'w') as f:
            json.dump(self.registry, f, indent=2)

    def register_model(self, model, version, architecture_desc, val_acc, params):
        """Speichert Modell mit Metadaten in der Registry."""
        timestamp  = datetime.now().isoformat()
        model_path = os.path.join(self.registry_dir, f'v{version}.keras')

        # Modell speichern
        model.save(model_path)
        model_size = os.path.getsize(model_path) / 1024

        # Metadaten
        metadata = {
            'version':      version,
            'timestamp':    timestamp,
            'architecture': architecture_desc,
            'val_accuracy': round(val_acc, 6),
            'n_parameters': params,
            'model_path':   model_path,
            'size_kb':      round(model_size, 1),
            'status':       'registered'
        }

        self.registry['models'].append(metadata)

        # Bestes Modell aktualisieren
        best = self.registry['best_model']
        if best is None or val_acc > best['val_accuracy']:
            self.registry['best_model'] = metadata
            metadata['status'] = 'champion'

        self._save_registry()
        print(f"  ✓ Version {version} registriert: Val-Acc={val_acc:.4f}, "
              f"Size={model_size:.1f}KB, Status={metadata['status']}")
        return metadata

    def load_best_model(self):
        """Lädt das beste registrierte Modell."""
        if self.registry['best_model'] is None:
            raise ValueError("Keine Modelle in der Registry!")
        best_meta  = self.registry['best_model']
        best_model = keras.models.load_model(best_meta['model_path'])
        print(f"\nBestes Modell geladen: Version {best_meta['version']}, "
              f"Val-Acc={best_meta['val_accuracy']:.4f}")
        return best_model, best_meta

    def print_table(self):
        """Zeigt alle registrierten Modelle als Tabelle."""
        print("\n" + "="*80)
        print("MODELL-REGISTRY")
        print("="*80)
        header = f"{'Ver':>4} | {'Val-Acc':>8} | {'Params':>10} | {'Size(KB)':>9} | {'Status':>12} | Architektur"
        print(header)
        print("-"*80)
        for m in sorted(self.registry['models'], key=lambda x: x['val_accuracy'], reverse=True):
            print(f"v{m['version']:>3} | {m['val_accuracy']:>8.4f} | {m['n_parameters']:>10,} | "
                  f"{m['size_kb']:>9.1f} | {m['status']:>12} | {m['architecture']}")

# ── 3. Drei verschiedene Modell-Versionen trainieren ──────
registry = ModelRegistry(MODEL_REGISTRY_DIR, METADATA_FILE)

architectures = [
    {'version': 1, 'desc': 'Klein: 64→32',       'units': [64, 32]},
    {'version': 2, 'desc': 'Mittel: 128→64',     'units': [128, 64]},
    {'version': 3, 'desc': 'Groß: 256→128→64', 'units': [256, 128, 64]},
]

print("Trainiere 3 Modellversionen...\n")
for arch in architectures:
    print(f"--- Version {arch['version']}: {arch['desc']} ---")

    # Modell bauen
    model_layers = [layers.Dense(arch['units'][0], activation='relu', input_shape=(784,))]
    for u in arch['units'][1:]:
        model_layers.append(layers.Dense(u, activation='relu'))
    model_layers.append(layers.Dense(10, activation='softmax'))
    model = keras.Sequential(model_layers, name=f"model_v{arch['version']}")
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    hist = model.fit(x_train, y_train, epochs=8, batch_size=256,
                      validation_split=0.1, verbose=0)
    val_acc = max(hist.history['val_accuracy'])

    registry.register_model(
        model=model,
        version=arch['version'],
        architecture_desc=arch['desc'],
        val_acc=val_acc,
        params=model.count_params()
    )

# ── 4. Bestes Modell laden und evaluieren ─────────────────
best_model, best_meta = registry.load_best_model()
test_loss, test_acc = best_model.evaluate(x_test, y_test, verbose=0)
print(f"Test-Accuracy: {test_acc:.4f}")

# ── 5. Versionstabelle ausgeben ───────────────────────────
registry.print_table()

# ── 6. Visualisierung ─────────────────────────────────────
models_info = registry.registry['models']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Modell-Versionierung: Vergleich', fontsize=13)

versions  = [f"v{m['version']}" for m in models_info]
val_accs  = [m['val_accuracy']  for m in models_info]
n_params  = [m['n_parameters']  for m in models_info]
sizes_kb  = [m['size_kb']       for m in models_info]
colors    = ['gold' if m['status'] == 'champion' else 'steelblue' for m in models_info]

ax = axes[0]
ax.bar(versions, val_accs, color=colors, alpha=0.85)
ax.set_title('Val-Accuracy (Gold = Champion)')
ax.set_ylabel('Val-Accuracy'); ax.grid(True, axis='y')
for i, (v, a) in enumerate(zip(versions, val_accs)):
    ax.text(i, a + 0.001, f'{a:.4f}', ha='center', fontsize=9)

ax2 = axes[1]
ax2.bar(versions, n_params, color=colors, alpha=0.85)
ax2.set_title('Modell-Größe (Parameter)')
ax2.set_ylabel('Parameter'); ax2.grid(True, axis='y')
for i, (v, p) in enumerate(zip(versions, n_params)):
    ax2.text(i, p + 100, f'{p:,}', ha='center', fontsize=9)

ax3 = axes[2]
ax3.scatter(n_params, val_accs, s=100, c=range(len(models_info)), cmap='viridis', zorder=5)
for i, (m) in enumerate(models_info):
    ax3.annotate(f"v{m['version']}", (m['n_parameters'], m['val_accuracy']),
                  textcoords='offset points', xytext=(5, 5), fontsize=10)
ax3.set_title('Parameter vs. Val-Accuracy (Effizienz)')
ax3.set_xlabel('Parameter'); ax3.set_ylabel('Val-Accuracy'); ax3.grid(True)

plt.tight_layout()
plt.savefig('F14_1_modell_versionierung.png', dpi=100)
plt.show()

print(f"\nBeste Version: {best_meta['version']} | {best_meta['architecture']}")
print(f"Test-Accuracy: {test_acc:.4f}")


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 14: Modell-Deployment
# Niveau: Fortgeschrittene
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import time
import csv
import os

np.random.seed(42)
tf.random.set_seed(42)

print("=== Batch Prediction Pipeline ===\n")

# ── 1. Modell trainieren ──────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax')
], name='batch_model')
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=5, batch_size=256, verbose=1)
print("Modell trainiert.\n")

# ── 2. Simulierte eingehende Samples ──────────────────────
N_SAMPLES  = 1000
BATCH_SIZE = 32
OUTPUT_CSV = 'batch_predictions.csv'

# Simuliere 1000 eingehende Samples (Mischung aus echten MNIST + zufälligen)
np.random.seed(123)
incoming_data   = x_test[:N_SAMPLES].copy()
incoming_ids    = np.arange(N_SAMPLES)
incoming_labels = y_test[:N_SAMPLES]  # Für spätere Evaluierung

# Einige Samples absichtlich korrumpieren (für Fehlerbehandlung)
corrupt_indices = np.random.choice(N_SAMPLES, 5, replace=False)
for idx in corrupt_indices:
    incoming_data[idx] = np.nan   # NaN einfügen = simulierter Fehler

print(f"Simuliere {N_SAMPLES} eingehende Samples")
print(f"  davon {len(corrupt_indices)} korrumpierte Samples (NaN)")
print(f"Batch-Größe: {BATCH_SIZE}")
print(f"Anzahl Batches: {int(np.ceil(N_SAMPLES / BATCH_SIZE))}\n")

# ── 3. Batch Prediction Pipeline ─────────────────────────
class BatchPredictionPipeline:
    """
    Verarbeitet eingehende Daten in Batches.
    Ergebnisse werden in CSV gespeichert.
    """

    def __init__(self, model, batch_size, output_csv):
        self.model      = model
        self.batch_size = batch_size
        self.output_csv = output_csv
        # Metriken
        self.total_processed = 0
        self.total_errors    = 0
        self.batch_times     = []
        self.batch_throughputs = []

    def preprocess(self, batch_data):
        """Vorverarbeitung: NaN-Check und Normalisierung."""
        clean_indices = []
        error_indices = []
        for i, sample in enumerate(batch_data):
            if np.any(np.isnan(sample)):
                error_indices.append(i)
            else:
                clean_indices.append(i)
        clean_data = batch_data[clean_indices] if clean_indices else np.empty((0, 784))
        return clean_data, clean_indices, error_indices

    def run(self, data, ids, save_csv=True):
        """Führt Batch-Prediction für alle Daten durch."""
        n_batches = int(np.ceil(len(data) / self.batch_size))
        all_results = []
        start_total = time.time()

        if save_csv:
            csv_file = open(self.output_csv, 'w', newline='')
            writer = csv.writer(csv_file)
            writer.writerow(['sample_id', 'prediction', 'confidence',
                              'status', 'batch_id', 'time_ms'])

        for batch_idx in range(n_batches):
            batch_start = time.time()
            start_i = batch_idx * self.batch_size
            end_i   = min(start_i + self.batch_size, len(data))

            batch_data = data[start_i:end_i]
            batch_ids  = ids[start_i:end_i]

            # Vorverarbeitung mit Fehlerbehandlung
            clean_data, clean_idx, error_idx = self.preprocess(batch_data)

            # Korrekte Samples verarbeiten
            if len(clean_data) > 0:
                try:
                    probs = self.model.predict(clean_data, verbose=0)
                    preds = np.argmax(probs, axis=1)
                    confs = probs.max(axis=1)
                except Exception as e:
                    print(f"  ⚠ Batch {batch_idx}: Fehler bei Inferenz: {e}")
                    preds = np.full(len(clean_data), -1)
                    confs = np.zeros(len(clean_data))

                for local_i, (global_i, pred, conf) in enumerate(
                        zip(clean_idx, preds, confs)):
                    result = {'id': int(batch_ids[global_i]), 'prediction': int(pred),
                               'confidence': round(float(conf), 4), 'status': 'ok',
                               'batch_id': batch_idx}
                    all_results.append(result)
                    if save_csv:
                        writer.writerow([result['id'], result['prediction'],
                                          result['confidence'], result['status'],
                                          result['batch_id'], ''])

            # Fehlerhafte Samples protokollieren
            for global_i in error_idx:
                result = {'id': int(batch_ids[global_i]), 'prediction': -1,
                           'confidence': 0.0, 'status': 'error_nan',
                           'batch_id': batch_idx}
                all_results.append(result)
                self.total_errors += 1
                if save_csv:
                    writer.writerow([result['id'], -1, 0.0, 'error_nan',
                                      batch_idx, ''])

            # Batch-Metriken
            batch_time = time.time() - batch_start
            n_in_batch = end_i - start_i
            throughput = n_in_batch / batch_time if batch_time > 0 else float('inf')
            self.batch_times.append(batch_time * 1000)
            self.batch_throughputs.append(throughput)
            self.total_processed += len(clean_data)

            if batch_idx % 5 == 0:
                print(f"  Batch {batch_idx+1:2d}/{n_batches}: "
                      f"{n_in_batch} Samples, {len(error_idx)} Fehler, "
                      f"{throughput:.1f} Samples/s, {batch_time*1000:.1f}ms")

        total_time = time.time() - start_total
        if save_csv:
            csv_file.close()

        print(f"\nVerarbeitung abgeschlossen:")
        print(f"  Total: {len(data)} Samples in {total_time:.2f}s")
        print(f"  Erfolgreich: {self.total_processed}")
        print(f"  Fehler: {self.total_errors}")
        print(f"  Gesamt-Throughput: {len(data)/total_time:.1f} Samples/s")
        print(f"  Mittlere Batch-Zeit: {np.mean(self.batch_times):.1f}ms")

        return all_results

# ── 4. Pipeline ausführen ─────────────────────────────────
pipeline = BatchPredictionPipeline(model, BATCH_SIZE, OUTPUT_CSV)
results  = pipeline.run(incoming_data, incoming_ids)

# ── 5. CSV-Datei anzeigen (erste Zeilen) ──────────────────
if os.path.exists(OUTPUT_CSV):
    print(f"\nErgebnisse gespeichert: {OUTPUT_CSV}")
    print("Erste 5 Zeilen:")
    with open(OUTPUT_CSV, 'r') as f:
        for i, line in enumerate(f):
            if i < 6:
                print(f"  {line.strip()}")

# ── 6. Genauigkeit auswerten ──────────────────────────────
correct = 0
valid   = 0
for r in results:
    if r['status'] == 'ok':
        valid += 1
        if r['prediction'] == int(incoming_labels[r['id']]):
            correct += 1
print(f"\nAccuracy auf validen Samples: {correct}/{valid} = {correct/valid:.4f}")

# ── 7. Visualisierung ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Batch Prediction Pipeline Metriken', fontsize=13)

# Batch-Zeit über Batches
ax = axes[0, 0]
ax.plot(pipeline.batch_times, color='blue', alpha=0.7)
ax.axhline(np.mean(pipeline.batch_times), color='red', linestyle='--',
            label=f'Mittelwert: {np.mean(pipeline.batch_times):.1f}ms')
ax.set_title('Batch-Zeit über alle Batches')
ax.set_xlabel('Batch-Index'); ax.set_ylabel('Zeit (ms)')
ax.legend(); ax.grid(True)

# Throughput
ax2 = axes[0, 1]
ax2.plot(pipeline.batch_throughputs, color='green', alpha=0.7)
ax2.axhline(np.mean(pipeline.batch_throughputs), color='red', linestyle='--',
             label=f'Mittelwert: {np.mean(pipeline.batch_throughputs):.0f} S/s')
ax2.set_title('Durchsatz (Samples/Sekunde)')
ax2.set_xlabel('Batch-Index'); ax2.set_ylabel('Samples/s')
ax2.legend(); ax2.grid(True)

# Status-Verteilung
ax3 = axes[1, 0]
status_counts = {}
for r in results:
    status_counts[r['status']] = status_counts.get(r['status'], 0) + 1
ax3.bar(status_counts.keys(), status_counts.values(),
         color=['green', 'red'], alpha=0.8)
ax3.set_title('Ergebnisstatistik')
ax3.set_ylabel('Anzahl'); ax3.grid(True, axis='y')

# Konfidenz-Verteilung
ax4 = axes[1, 1]
confs = [r['confidence'] for r in results if r['status'] == 'ok']
ax4.hist(confs, bins=20, color='steelblue', edgecolor='white')
ax4.set_title(f'Konfidenz-Verteilung (n={len(confs)})')
ax4.set_xlabel('Konfidenz'); ax4.set_ylabel('Anzahl'); ax4.grid(True)
ax4.axvline(np.mean(confs), color='red', linestyle='--',
             label=f'Mittelwert: {np.mean(confs):.3f}')
ax4.legend()

plt.tight_layout()
plt.savefig('F14_2_batch_prediction_pipeline.png', dpi=100)
plt.show()


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 14: Modell-Deployment
# Niveau: Fortgeschrittene
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from scipy.special import kl_div

np.random.seed(42)
tf.random.set_seed(42)

print("=== Modell-Monitoring: Data Drift Erkennung ===\n")

# ── 1. Modell trainieren ──────────────────────────────────
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(784,)),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax')
], name='monitored_model')
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=5, batch_size=256, verbose=1)
print("Modell trainiert.\n")

# ── 2. Referenz-Verteilung (keine Drift) ─────────────────
# Die Baseline aus Training-Daten ermitteln
ref_preds  = model.predict(x_train[:1000], verbose=0)
ref_pred_dist = np.bincount(np.argmax(ref_preds, axis=1), minlength=10) / 1000
ref_conf_mean = ref_preds.max(axis=1).mean()
ref_pixel_mean = x_train[:1000].mean()

print("Referenz-Verteilung (Baseline):")
print(f"  Pred-Verteilung: {np.round(ref_pred_dist, 3)}")
print(f"  Mittlere Konfidenz: {ref_conf_mean:.3f}")
print(f"  Mittlerer Pixelwert: {ref_pixel_mean:.3f}\n")

# ── 3. Data Drift simulieren ──────────────────────────────
N_BATCHES  = 20          # Anzahl überwachter Batches
BATCH_SIZE = 200
DRIFT_START = 8          # Ab Batch 8 beginnt Drift

# KL-Divergenz-Schwellenwert für Alarm
KL_THRESHOLD = 0.15

def add_drift(data, drift_strength):
    """
    Fügt graduellen Drift hinzu:
    - Erhöht Helligkeit (simuliert Kamera-Drift)
    - Fügt Rauschen hinzu
    """
    drifted = data + drift_strength * np.random.randn(*data.shape) * 0.1
    drifted = drifted + drift_strength * 0.2  # Helligkeits-Drift
    return np.clip(drifted, 0, 1)

def compute_kl_divergence(p, q):
    """KL-Divergenz: KL(P || Q) für Vorhersageverteilungen."""
    p = np.array(p) + 1e-10
    q = np.array(q) + 1e-10
    p = p / p.sum()
    q = q / q.sum()
    return np.sum(p * np.log(p / q))

# ── 4. Monitoring-Schleife ────────────────────────────────
print("Simuliere Monitoring über 20 Batches...")
print("(Drift beginnt ab Batch 8)\n")

monitoring_log = {
    'batch_id':       [],
    'kl_divergence':  [],
    'mean_confidence':[],
    'pixel_mean':     [],
    'alerts':         [],
    'drift_strength': []
}

for batch_id in range(N_BATCHES):
    # Datendrift graduell einführen
    drift_strength = max(0, (batch_id - DRIFT_START + 1) * 0.3) if batch_id >= DRIFT_START else 0.0

    # Eingehende Daten simulieren (mit Drift)
    idx = np.random.choice(len(x_test), BATCH_SIZE, replace=False)
    raw_batch = x_test[idx]
    drifted_batch = add_drift(raw_batch, drift_strength)

    # Vorhersagen
    preds = model.predict(drifted_batch, verbose=0)
    pred_dist = np.bincount(np.argmax(preds, axis=1), minlength=10) / BATCH_SIZE
    mean_conf = preds.max(axis=1).mean()
    pixel_mean = drifted_batch.mean()

    # KL-Divergenz zur Referenz
    kl = compute_kl_divergence(pred_dist, ref_pred_dist)

    # Alarm?
    alert = kl > KL_THRESHOLD
    monitoring_log['batch_id'].append(batch_id)
    monitoring_log['kl_divergence'].append(kl)
    monitoring_log['mean_confidence'].append(mean_conf)
    monitoring_log['pixel_mean'].append(pixel_mean)
    monitoring_log['alerts'].append(alert)
    monitoring_log['drift_strength'].append(drift_strength)

    alert_str = " 🚨 DRIFT ALERT!" if alert else ""
    print(f"Batch {batch_id+1:2d}: KL={kl:.4f}, Konfidenz={mean_conf:.3f}, "
          f"Pixel-Mean={pixel_mean:.3f}, Drift={drift_strength:.1f}{alert_str}")

# ── 5. Alert-Zusammenfassung ──────────────────────────────
n_alerts = sum(monitoring_log['alerts'])
first_alert = next((i for i, a in enumerate(monitoring_log['alerts']) if a), None)

print(f"\n{'='*50}")
print(f"MONITORING ZUSAMMENFASSUNG:")
print(f"  Alarme ausgelöst: {n_alerts}/{N_BATCHES}")
print(f"  Erster Alarm: Batch {first_alert + 1 if first_alert is not None else 'Kein'}")
print(f"  KL-Schwellenwert: {KL_THRESHOLD}")
print(f"  Drift begann: Batch {DRIFT_START + 1}")
if first_alert is not None:
    print(f"  Erkennungs-Verzögerung: {first_alert - DRIFT_START} Batches")

# ── 6. Visualisierung ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Modell-Monitoring: Data Drift Erkennung', fontsize=14)

batches = monitoring_log['batch_id']

# KL-Divergenz über Zeit
ax = axes[0, 0]
ax.plot(batches, monitoring_log['kl_divergence'], marker='o', color='blue', label='KL-Divergenz')
ax.axhline(KL_THRESHOLD, color='red', linestyle='--', linewidth=2,
            label=f'Schwellenwert ({KL_THRESHOLD})')
ax.axvline(DRIFT_START - 0.5, color='orange', linestyle=':',
            label='Drift startet')
# Alarm-Punkte
alert_batches = [b for b, a in zip(batches, monitoring_log['alerts']) if a]
alert_kls     = [monitoring_log['kl_divergence'][b] for b in alert_batches]
if alert_batches:
    ax.scatter(alert_batches, alert_kls, color='red', s=100, zorder=5, label='ALERT')
ax.set_title('KL-Divergenz (Vorhersageverteilung)')
ax.set_xlabel('Batch'); ax.set_ylabel('KL-Divergenz')
ax.legend(fontsize=8); ax.grid(True)

# Mittlere Konfidenz über Zeit
ax2 = axes[0, 1]
ax2.plot(batches, monitoring_log['mean_confidence'], marker='s', color='green',
          label='Mittl. Konfidenz')
ax2.axhline(ref_conf_mean, color='blue', linestyle='--',
             label=f'Referenz: {ref_conf_mean:.3f}')
ax2.axvline(DRIFT_START - 0.5, color='orange', linestyle=':')
ax2.set_title('Mittlere Vorhersage-Konfidenz')
ax2.set_xlabel('Batch'); ax2.set_ylabel('Konfidenz')
ax2.legend(); ax2.grid(True)

# Pixelmittelwert über Zeit
ax3 = axes[1, 0]
ax3.plot(batches, monitoring_log['pixel_mean'], marker='^', color='purple',
          label='Pixel-Mittelwert')
ax3.axhline(ref_pixel_mean, color='blue', linestyle='--',
             label=f'Referenz: {ref_pixel_mean:.3f}')
ax3.axvline(DRIFT_START - 0.5, color='orange', linestyle=':')
ax3.fill_between(batches,
                  [ref_pixel_mean - 0.05]*len(batches),
                  [ref_pixel_mean + 0.05]*len(batches),
                  alpha=0.2, color='blue', label='±0.05 Toleranz')
ax3.set_title('Input-Verteilung: Pixel-Mittelwert')
ax3.set_xlabel('Batch'); ax3.set_ylabel('Mittlerer Pixelwert')
ax3.legend(fontsize=8); ax3.grid(True)

# Drift-Stärke vs. KL-Divergenz
ax4 = axes[1, 1]
scatter = ax4.scatter(monitoring_log['drift_strength'],
                       monitoring_log['kl_divergence'],
                       c=batches, cmap='viridis', s=60)
plt.colorbar(scatter, ax=ax4, label='Batch-Nr.')
ax4.axhline(KL_THRESHOLD, color='red', linestyle='--',
             label=f'KL-Schwellenwert ({KL_THRESHOLD})')
ax4.set_title('Drift-Stärke vs. KL-Divergenz')
ax4.set_xlabel('Drift-Stärke'); ax4.set_ylabel('KL-Divergenz')
ax4.legend(); ax4.grid(True)

plt.tight_layout()
plt.savefig('F14_3_modell_monitoring.png', dpi=100)
plt.show()

print("\nMonitoring-Simulation abgeschlossen!")
